# Module 2: Topics & Partitions

本章目标：
- 理解 Partition 如何实现消息的并行写入与有序存储
- 掌握消息 **Key** 对分区路由的决定作用
- 观察多 Partition Topic 下 offset 的分布规律
- 学会用 Admin API 查询 Topic 元数据

---

## 前置条件

- 已完成 Module 1，集群正在运行（`docker compose up -d`）
- `BROKERS = "localhost:19094,localhost:29094,localhost:39094"`

## 核心概念

### 为什么需要 Partition？

一个 Topic 内的消息若都存在单个节点上，写入吞吐受单机 I/O 限制。
Partition 将 Topic 拆分为多个有序日志，分布在不同 Broker 上，实现**水平扩展**：

```
Topic: orders  (3 partitions)

Partition 0: [offset 0] [offset 1] [offset 2] ...   → Broker 1
Partition 1: [offset 0] [offset 1] [offset 2] ...   → Broker 2
Partition 2: [offset 0] [offset 1] [offset 2] ...   → Broker 3
```

### 消息路由规则

| 场景 | 路由方式 |
|------|----------|
| 无 key | Round-robin（轮询各分区）|
| 有 key | `murmur2_hash(key) % num_partitions`（相同 key 总进同一分区）|
| 指定 partition | 直接写入指定分区 |

> **注意**：Key 的分区映射取决于 murmur2 哈希函数，不同 key 不保证均匀分布。
> 使用 key 的核心价值是**保证相同 key 的消息严格有序**，而非均衡分布。

In [ ]:
import asyncio
from aiokafka import AIOKafkaProducer, AIOKafkaConsumer, TopicPartition
from aiokafka.admin import AIOKafkaAdminClient, NewTopic
from aiokafka.partitioner import DefaultPartitioner
from collections import defaultdict

BROKERS = "localhost:19094,localhost:29094,localhost:39094"
TOPIC = "partitions-demo"
NUM_PARTITIONS = 3

## 0. 预览：Key 的分区映射

先了解不同 key 会路由到哪个分区（由 murmur2 hash 决定）：

In [ ]:
# 预览 key → partition 映射（不发送消息，仅计算）
partitioner = DefaultPartitioner()
partitions = list(range(NUM_PARTITIONS))

test_keys = ["alice", "bob", "charlie", "dave", "eve"]
print(f"Key → Partition 映射（{NUM_PARTITIONS} partitions, murmur2 hash）:")
for k in test_keys:
    p = partitioner(k.encode(), partitions, partitions)
    print(f"  {k!r:12} → partition {p}")

print("\n注意：alice/bob/charlie 都落在 P0；dave→P1, eve→P2")
print("这不是 bug，而是 murmur2 hash 的结果——均匀分布需要更多 key")

## 1. 创建多分区 Topic

In [ ]:
async def recreate_topic(brokers, topic, num_partitions, replication_factor=3):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        existing = await admin.list_topics()
        if topic in existing:
            await admin.delete_topics([topic])
            await asyncio.sleep(1)  # 等待删除完成
        await admin.create_topics([
            NewTopic(name=topic, num_partitions=num_partitions, replication_factor=replication_factor)
        ])
        print(f"✓ Topic '{topic}' 创建成功（{num_partitions} 个分区）")
        
        descriptions = await admin.describe_topics([topic])
        for desc in descriptions:
            for p in sorted(desc['partitions'], key=lambda x: x['partition']):
                print(f"  P{p['partition']}: leader={p['leader']}, replicas={p['replicas']}, isr={p['isr']}")
    finally:
        await admin.close()

await recreate_topic(BROKERS, TOPIC, NUM_PARTITIONS)

## 2. 无 Key 发送：观察 Round-robin 分布

In [ ]:
async def produce_no_key(brokers, topic, count=9):
    producer = AIOKafkaProducer(bootstrap_servers=brokers)
    await producer.start()
    partition_counts = defaultdict(int)
    try:
        for i in range(count):
            meta = await producer.send_and_wait(topic, f"msg-{i:02d}".encode())
            partition_counts[meta.partition] += 1
            print(f"  msg-{i:02d} → partition={meta.partition}, offset={meta.offset}")
    finally:
        await producer.stop()
    print(f"\n分区分布: {dict(sorted(partition_counts.items()))}")

print("=== 无 Key 发送（round-robin）===")
await produce_no_key(BROKERS, TOPIC)

## 3. 带 Key 发送：相同 Key 总进同一分区

选择哈希到不同分区的 key，直观展示分区路由：
- `alice` → P0
- `dave`  → P1
- `eve`   → P2

典型业务场景：同一用户的事件保证有序（key = user_id）。

In [ ]:
async def produce_with_key(brokers, topic):
    producer = AIOKafkaProducer(bootstrap_servers=brokers)
    await producer.start()
    try:
        # 这三个 key 经 murmur2 hash 分别落在 P0, P1, P2
        users = [("alice", "P0"), ("dave", "P1"), ("eve", "P2")]
        for user, expected in users:
            for i in range(3):
                msg = f"{user}: event-{i}"
                meta = await producer.send_and_wait(
                    topic,
                    key=user.encode(),
                    value=msg.encode(),
                )
                print(f"  key={user!r:8} (→{expected}) | partition={meta.partition}, offset={meta.offset}")
    finally:
        await producer.stop()

print("=== 带 Key 发送 ===")
await produce_with_key(BROKERS, TOPIC)
print("\n观察：相同 key 始终路由到同一分区")

## 4. 按分区消费：观察 offset 独立计数

In [ ]:
async def inspect_partitions(brokers, topic, num_partitions):
    """使用 getmany() 读取各分区现有消息（避免 consumer_timeout_ms 的兼容性问题）。"""
    consumer = AIOKafkaConsumer(
        bootstrap_servers=brokers,
        auto_offset_reset="earliest",
    )
    tps = [TopicPartition(topic, p) for p in range(num_partitions)]
    consumer.assign(tps)
    await consumer.start()
    partition_msgs = defaultdict(list)
    try:
        # idle 超过 1s 认为已读完所有现有消息
        while True:
            results = await consumer.getmany(*tps, timeout_ms=1000, max_records=100)
            if not results:
                break
            for tp, msgs in results.items():
                for msg in msgs:
                    partition_msgs[msg.partition].append((msg.offset, msg.key, msg.value.decode()))
    finally:
        await consumer.stop()

    for p in sorted(partition_msgs):
        print(f"\nPartition {p} ({len(partition_msgs[p])} 条消息):")
        for offset, key, value in partition_msgs[p]:
            key_str = key.decode() if key else None
            print(f"  offset={offset:3d} | key={key_str!r:8} | {value}")

await inspect_partitions(BROKERS, TOPIC, NUM_PARTITIONS)

## 5. 查询 Topic 元数据

In [ ]:
async def describe_topic(brokers, topic):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        descriptions = await admin.describe_topics([topic])
        for desc in descriptions:
            print(f"Topic: {desc['topic']}")
            for p in sorted(desc['partitions'], key=lambda x: x['partition']):
                print(f"  Partition {p['partition']}: leader={p['leader']}, replicas={p['replicas']}, isr={p['isr']}")
    finally:
        await admin.close()

await describe_topic(BROKERS, TOPIC)

## 总结

- **分区** 是 Kafka 并行能力的核心，每个分区是独立的有序日志
- **无 Key** 消息轮询写入各分区，吞吐高但无顺序保证
- **有 Key** 消息路由到固定分区（基于 murmur2 hash），同 Key 消息严格有序
- 每个分区的 offset 独立从 0 开始累积
- `leader` 负责该分区的读写，`replicas` 是副本集，`isr` 是同步副本集

---

## 下一章

**Module 3: Consumer Groups** — 多个 Consumer 如何协同消费，Partition 如何在 Consumer 间分配，以及 Rebalance 触发时机。